# Chapter 2: Build GPT From Scratch

Character-level GPT trained on tiny-Shakespeare. Every stage builds on the last — run cells in order.

See `../RUNBOOK.md` for the why/enterprise-notes/interview-angles, and `WALKTHROUGH.md` in this folder for the deep-dive line-by-line breakdown. Code cells below are heavily commented so the notebook itself is a usable reference too, not just a script to run.

## Stage 1: Data loading + character-level tokenization

In [ ]:
# Download the tiny Shakespeare dataset (~1MB, ~1.1M characters).
# -nc = "no clobber": skip re-downloading if input.txt already exists in this directory.
!wget -nc https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

In [ ]:
# Read the entire file into memory as one big Python string.
# At this scale (~1MB) that's totally fine; real corpora (Chapter 4) get streamed/chunked instead.
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

print(f"Dataset length: {len(text):,} characters")
print("\nFirst 300 characters:\n")
print(text[:300])

### Build the vocabulary
Find every unique character in the dataset — that's our entire vocabulary. No subwords, no BPE merges yet (that's Chapter 4).

In [ ]:
# set(text) collapses the string down to its unique characters (order lost);
# sorted(...) gives us a deterministic, reproducible ordering -> the same character
# always gets the same integer ID across runs, which matters for reproducibility.
chars = sorted(list(set(text)))

# vocab_size is the single most important number in this notebook -- it determines
# the width of every embedding table and every output layer we build from here on.
vocab_size = len(chars)

print(f"Vocabulary size: {vocab_size}")
print(f"All characters: {''.join(chars)!r}")  # includes letters, punctuation, newline, space, etc.

### Encoder / decoder
`stoi` (string-to-int) and `itos` (int-to-string) are the entire tokenizer. `encode` and `decode` are just dictionary lookups.

In [ ]:
# Two dictionaries are our ENTIRE tokenizer -- this is the whole "tokenization" step
# for a character-level model. Compare this to Chapter 4, where we'll train a real
# BPE tokenizer with thousands of learned subword merges instead of ~65 raw characters.
stoi = {ch: i for i, ch in enumerate(chars)}   # e.g. {'a': 40, 'b': 41, ...}
itos = {i: ch for i, ch in enumerate(chars)}   # e.g. {40: 'a', 41: 'b', ...}

# encode: string -> list[int]  (look up each character's integer ID)
encode = lambda s: [stoi[c] for c in s]
# decode: list[int] -> string  (look up each ID's character, then join them back together)
decode = lambda l: ''.join([itos[i] for i in l])

# Round-trip sanity check: encoding then decoding should return the exact original string.
test = "Hello"
encoded = encode(test)
print(f"{test!r} encoded: {encoded}")
print(f"Decoded back: {decode(encoded)!r}")

### Encode the entire dataset into a tensor

In [ ]:
import torch

# Encode the WHOLE dataset in one shot: every character in the 1.1M-character
# corpus becomes one integer. dtype=torch.long because these are indices into an
# embedding table -- PyTorch's embedding/index ops require integer (long) tensors.
data = torch.tensor(encode(text), dtype=torch.long)

print(f"Data shape: {data.shape}, dtype: {data.dtype}")
print(f"First 100 encoded characters:\n{data[:100]}")

### Train / validation split (90 / 10)
Never evaluate on data the model trained on — validation loss is our only honest signal for overfitting.

In [ ]:
# Simple contiguous split (first 90% / last 10%) rather than a random shuffle --
# this is a text-continuation task, so we want validation data the model has
# genuinely never seen any part of, not just individually-shuffled characters
# from the same passages it trained on.
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

print(f"Training characters:   {len(train_data):,}")
print(f"Validation characters: {len(val_data):,}")

### Context windows (`block_size`)
The model never sees the whole dataset at once — only a fixed-size window. One window of length `block_size` actually yields `block_size` separate training examples (predict the next char at every position).

In [ ]:
# block_size = the context window / how many characters of history the model can see
# at once. Small on purpose here (8) so every individual training example below can
# be printed and inspected by eye. GPT-2 uses 1024; modern LLMs use tens of thousands.
block_size = 8

# x = the input characters, y = x shifted one position to the right (the "next char"
# targets). This single offset trick is how autoregressive next-token training works.
x = train_data[:block_size]
y = train_data[1:block_size + 1]

print("One block of text yields these context -> target pairs:\n")
for t in range(block_size):
    context = x[:t+1]        # everything seen so far, growing one character at a time
    target = y[t]             # the single character that should come next
    print(f"context: {decode(context.tolist())!r:20} -> target: {decode([target.item()])!r}")

### Batching
Stack many independent context windows together so the GPU processes them in parallel instead of one at a time.

In [ ]:
# Fix the random seed so results are reproducible across runs/machines -- useful when
# comparing loss curves later, since "did my change help" is only meaningful if the
# random batches being compared are otherwise identical.
torch.manual_seed(1337)

batch_size = 4  # how many independent sequences we process simultaneously (this is "B")

def get_batch(split):
    """Sample a random batch of (input, target) sequence pairs from train or val data."""
    data_split = train_data if split == 'train' else val_data

    # Pick batch_size random starting positions. Upper bound is
    # len(data_split) - block_size so every window of length block_size stays in-bounds.
    ix = torch.randint(len(data_split) - block_size, (batch_size,))

    # For each starting position i, grab a block_size-length window as input,
    # and the SAME window shifted one character later as the target -- exactly
    # the same offset trick as the single-example version above, just batched.
    x = torch.stack([data_split[i:i+block_size] for i in ix])
    y = torch.stack([data_split[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:', xb.shape)   # (batch_size, block_size) = (4, 8)
print(xb)
print('targets:', yb.shape)  # same shape -- one target character per input position
print(yb)

**Checkpoint:** at this point you have `get_batch('train')` / `get_batch('val')` producing `(batch_size, block_size)` integer tensors ready to feed into embeddings. Next: the Bigram Language Model — the simplest possible baseline, built before self-attention so we can *feel* why it fails.

## Stage 2: The Bigram Language Model (baseline)

See `WALKTHROUGH.md` in this folder for the full line-by-line breakdown, shape tracing, and the guide-dog/blind-hiker training analogy.

This is deliberately the *dumbest possible* language model: given one character, predict the next, with zero memory of anything before it. We build and train it first so the self-attention section (next) has something concrete to improve on.

In [ ]:
import torch.nn as nn
from torch.nn import functional as F

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # This single embedding table IS the entire model. Row i holds vocab_size
        # raw scores ("logits") for what character comes after character i.
        # There's no hidden layer, no mixing of information across positions --
        # each character's prediction depends ONLY on that one character.
        # Shape: (vocab_size, vocab_size), e.g. 65x65 for our Shakespeare vocab.
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        # idx: (B, T) integer tensor of character indices.
        # Looking up idx in the embedding table gives, for every position, that
        # character's row of "next character" scores.
        logits = self.token_embedding_table(idx)  # (B, T, C) where C = vocab_size

        if targets is None:
            # Inference/generation mode: no ground truth to compare against, so skip loss.
            loss = None
        else:
            # F.cross_entropy expects predictions as (N, C) and targets as (N,) --
            # i.e. a flat list of examples, not a (B, T, C) 3D tensor. Flattening
            # collapses batch and time together: every (sequence, position) pair
            # becomes one independent training example.
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        """Autoregressively sample max_new_tokens new characters, one at a time."""
        # idx starts as (B, T) -- whatever context we're continuing from.
        for _ in range(max_new_tokens):
            logits, loss = self(idx)          # forward pass on everything generated so far
            logits = logits[:, -1, :]         # bigram model only uses the LAST character's
                                                # prediction; shape (B, T, C) -> (B, C)
            probs = F.softmax(logits, dim=-1) # convert raw scores to a probability distribution
            idx_next = torch.multinomial(probs, num_samples=1)  # weighted random sample, (B, 1)
            idx = torch.cat((idx, idx_next), dim=1)  # append the sampled character; loop continues
        return idx

model = BigramLanguageModel(vocab_size)
logits, loss = model(xb, yb)
print(f"Logits shape: {logits.shape}")           # (B*T, vocab_size) after the internal flatten
print(f"Initial loss: {loss.item():.4f}")
# Sanity check: an untrained model's weights are random, so it's effectively guessing
# uniformly at random over the vocabulary. This computes the cross-entropy loss that
# PURE random guessing would produce -- if our actual initial loss doesn't roughly
# match this, something's wrong with initialization before we've spent any compute training.
print(f"Expected loss for random guessing: {-torch.log(torch.tensor(1.0/vocab_size)):.4f}")

If `Initial loss` is close to `Expected loss`, the model is correctly initialized (i.e. currently guessing uniformly at random over the vocabulary — it hasn't learned anything yet).

In [ ]:
# Generate from the UNTRAINED model -- expect pure gibberish, since the embedding
# table is still random weights at this point.
# Start from a single "newline" token (index 0) as the seed context: shape (1, 1) = (B=1, T=1).
context = torch.zeros((1, 1), dtype=torch.long)
generated = model.generate(context, max_new_tokens=100)
print(decode(generated[0].tolist()))  # generated[0] strips the batch dimension before decoding

### Train the Bigram model

In [ ]:
# AdamW: the standard optimizer for nearly all modern deep learning, including
# GPT-4-class models. Combines momentum (smooths past bumps in the loss landscape),
# per-parameter adaptive learning rates (rare characters get bigger updates than
# common ones), and weight decay (keeps weights small/stable, reduces overfitting).
# lr=1e-3 is a reasonably aggressive learning rate, fine for a model this tiny/simple.
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

batch_size = 32  # larger batch than our earlier demo (4) -- more stable gradient estimates

for steps in range(10000):
    xb, yb = get_batch('train')          # fresh random batch every step

    logits, loss = model(xb, yb)         # forward pass: predict + compute loss vs. ground truth

    # PyTorch accumulates gradients by default (adds new ones onto old ones), so we
    # must explicitly clear them before each backward pass -- otherwise gradients
    # from previous steps would corrupt this step's update.
    optimizer.zero_grad(set_to_none=True)

    loss.backward()                      # compute gradients: how would nudging each
                                           # weight up/down change the loss?
    optimizer.step()                     # apply those gradients to actually update the weights

    if steps % 1000 == 0:
        print(f"Step {steps}: loss = {loss.item():.4f}")

print(f"\nFinal loss: {loss.item():.4f}")

In [ ]:
# Generate from the TRAINED model -- better than before, but still poor: the model
# has learned common CHARACTER-PAIR statistics (e.g. 't' is often followed by 'h'),
# but has no memory of anything beyond the single most recent character.
context = torch.zeros((1, 1), dtype=torch.long)
generated = model.generate(context, max_new_tokens=500)
print(decode(generated[0].tolist()))

**Checkpoint:** loss should drop from ~4.7 (random) to ~2.4-2.5. The output is *slightly* more English-shaped than noise but still gibberish — because the bigram model only ever looks at the single most recent character. It can't tell whether `h` came from `t` (predict `e`), `s` (predict `i`), or something else. That's exactly the problem self-attention solves next: letting characters look back at everything before them, not just the last one.

**Stop here and run this section before continuing** — confirm your loss curve and generated text look like the above, then we move to Part 9: the mathematical trick behind self-attention.

## Stage 3: The Mathematical Trick Behind Self-Attention

See `WALKTHROUGH.md` for the full breakdown. Core idea: self-attention starts as nothing more exotic
than "let each token average in the tokens before it." The Q/K/V machinery (next stage) is just a
smarter, *learned* way of deciding those averaging weights -- but the mechanical trick underneath is
exactly the matrix multiplication built here.

We build the SAME averaging operation three ways, each version replacing the last, to build up to the
operation self-attention will reuse directly.

In [ ]:
torch.manual_seed(42)

# Small toy tensor, deliberately abstract (not characters) -- this trick applies to ANY
# per-token feature vectors, which is exactly what token+position embeddings will be
# once we build the real GPT model. B=batch, T=time(tokens), C=channels(features per token).
B, T, C = 4, 8, 2
x = torch.randn(B, T, C)
print(f"x shape: {x.shape}  (B={B} sequences, T={T} tokens each, C={C} features per token)")

### Version 1: naive loop
The goal: for every token, replace it with the AVERAGE of itself and every token before it (never
tokens after -- that would leak future information the model shouldn't have access to at that position).
This is the crudest possible form of "tokens talking to each other."

In [ ]:
# xbow = "x, bag-of-words style average" -- each position becomes the running average
# of everything up to and including itself.
xbow = torch.zeros((B, T, C))
for b in range(B):          # for every sequence in the batch...
    for t in range(T):      # for every time step in that sequence...
        xprev = x[b, :t+1]              # all tokens from position 0 through t (inclusive)
        xbow[b, t] = torch.mean(xprev, dim=0)  # collapse them into one averaged vector

print("Using loops -- first sequence's averaged tokens:")
print(xbow[0])
print("\nWorks correctly, but this is TWO nested Python loops -- painfully slow on real")
print("data (imagine T=1024, B=64). We need the same math, vectorized.")

### Version 2: the same result via matrix multiplication
The key insight: a **lower-triangular matrix of 1s, row-normalized**, IS an averaging operation when
multiplied against your data. Row t has 1s in columns 0..t (meaning "I can see these") and 0s after
(meaning "future, invisible"). Normalizing each row to sum to 1 turns "can see" into "equal-weighted
average over everything I can see."

In [ ]:
# torch.tril = lower TRIanguLar: keeps values on/below the diagonal, zeroes out everything
# above it. This is the mask that enforces "no looking at the future" -- the same causal
# constraint we already saw in the bigram model's generate(), now expressed as a matrix.
wei = torch.tril(torch.ones(T, T))
print("Raw lower-triangular matrix (1 = 'can see this position', 0 = 'future, blocked'):")
print(wei)

# Row-normalize: divide each row by its own sum, so each row's 1s become equal fractions
# that sum to 1. Row 0 -> [1.0, 0, 0, ...] (only itself). Row 3 -> [0.25,0.25,0.25,0.25,0,...]
# (equal-weighted average of positions 0-3). This row IS the "attention weights" for that position.
wei = wei / wei.sum(1, keepdim=True)
print("\nAfter row-normalizing (each row now sums to 1 -- an averaging weight matrix):")
print(wei)

# THE key operation: (T, T) @ (B, T, C) -> (B, T, C). PyTorch broadcasts the 2D weight
# matrix across the batch dimension automatically. This single matmul replaces BOTH
# nested loops above -- and runs as one fast, parallelized GPU operation.
xbow2 = wei @ x
print("\nxbow2[0] (via wei @ x):")
print(xbow2[0])
print(f"\nIdentical to the loop version? {torch.allclose(xbow, xbow2)}")

**Key insight: matrix multiplication can compute weighted averages in parallel.** This is the
mechanical trick self-attention is built on -- everything from here is about making the WEIGHTS in
that matrix smarter (learned, data-dependent) rather than a fixed "equal average of everything so far."

### Version 3: softmax-based masking
One more reformulation, functionally identical to Version 2, but structured the way real self-attention
will actually compute it: start from raw SCORES (all zero here, for now), mask out the future with
`-inf`, then use softmax to turn scores into normalized weights. `-inf` matters because
`softmax(-inf) = 0` exactly -- it's how masking gets baked directly into the probability distribution
instead of needing a separate normalization step afterward.

In [ ]:
tril = torch.tril(torch.ones(T, T))   # same causal mask as before

# Start from all-zero "affinity scores" between positions -- in real self-attention
# (next stage) these zeros get replaced by learned Query-Key dot products; here they're
# deliberately zero so this version stays mathematically equivalent to Version 1 and 2.
wei3 = torch.zeros((T, T))

# masked_fill: wherever tril==0 (future positions), overwrite the score with -infinity.
# This is the mechanism that enforces "cannot attend to the future" once scores stop
# being uniform zeros -- crucial once real Q/K attention scores are plugged in next.
wei3 = wei3.masked_fill(tril == 0, float('-inf'))
print("Scores after masking (future positions set to -inf):")
print(wei3)

# softmax converts scores to probabilities that sum to 1 per row. exp(-inf) = 0, so
# masked positions get EXACTLY zero probability -- softmax handles the masking for us.
wei3 = F.softmax(wei3, dim=-1)
print("\nAfter softmax (-inf -> exactly 0 probability; remaining scores -> uniform since")
print("they started equal):")
print(wei3)

xbow3 = wei3 @ x
print(f"\nSame result as Version 1 and 2? {torch.allclose(xbow, xbow3)}")

**Checkpoint:** all three versions produce identical output -- they're the same operation expressed
three ways. Version 3's structure (raw scores -> mask future with `-inf` -> softmax -> matmul with
values) is EXACTLY the structure real self-attention uses in Part 10 -- the only thing that changes is
where the raw scores come from. Instead of "all zeros" (forcing a uniform average), Part 10 computes
them from learned Query and Key projections of the data itself -- so the model can learn WHICH past
tokens matter most for predicting the next one, rather than averaging all of them equally.

**Stop here and run this section before continuing** -- confirm all three `torch.allclose(...)` checks
print `True`, then we move to Part 10: implementing real self-attention with learned Query, Key, and
Value projections.

## Stage 4: Implementing Self-Attention (Part 10)

See `WALKTHROUGH.md` for the Query/Key/Value intuition and a small hand-traceable example. Here's the
one-sentence version of what changes from Stage 3: **the "scores" that used to be hardcoded zeros are
now computed from the data itself**, via three separate learned linear layers -- Query, Key, and Value.
Everything downstream (mask, softmax, weighted sum) is IDENTICAL machinery to Version 3 in Stage 3.

In [ ]:
torch.manual_seed(1337)

# Bigger embedding dimension than Stage 3's toy C=2 -- this now matches what a real
# token embedding will look like once we build the full GPT model (Part 12).
B, T, C = 4, 8, 32
x = torch.randn(B, T, C)

# head_size is a DESIGN CHOICE, independent of C -- it's how wide the Query/Key/Value
# vectors are. WHY smaller than C: (1) cheaper compute -- the score matrix involves
# vectors of length head_size, so smaller head_size = less work per pair, which matters
# a lot once T is in the thousands; (2) it sets up Part 11's multi-head attention -- one
# projection comparing all C traits at once is worse than several SMALLER, independent
# projections each free to specialize in a different kind of relevance (like several
# focused matchmakers instead of one generalist). Real models often set
# head_size = C / num_heads so multiple heads' outputs concatenate back to C.
head_size = 16

# Three SEPARATE learned linear layers, no bias. Each is a (C, head_size) weight matrix
# that projects a token's C-dim embedding down to a head_size-dim vector.
# bias=False is standard for attention projections -- keeps the operation a pure
# linear transformation of the embedding, nothing added independent of the input.
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

# Apply each projection to every token in the batch simultaneously.
k = key(x)     # (B, T, head_size) -- "what does each token CONTAIN?"
q = query(x)   # (B, T, head_size) -- "what is each token LOOKING FOR?"
v = value(x)   # (B, T, head_size) -- "what does each token COMMUNICATE if attended to?"
print(f"k shape: {k.shape}, q shape: {q.shape}, v shape: {v.shape}")

### Compute attention scores ("affinities")
This replaces Stage 3's hardcoded `torch.zeros((T, T))` starting scores with real, data-dependent
values -- the dot product between every token's Query and every OTHER token's Key.

In [ ]:
# q @ k.transpose(-2, -1): (B, T, head_size) @ (B, head_size, T) -> (B, T, T)
# Every row t is "token t's query, dotted against every token's key" -- a high dot
# product means "this key strongly matches what I'm looking for," i.e. high relevance.
wei = q @ k.transpose(-2, -1)  # (B, T, T) -- one score per (query token, key token) pair

# Scale by 1/sqrt(head_size). WHY: without this, as head_size grows, dot products grow
# in magnitude too (summing more terms), which pushes softmax into a very "peaked"
# regime (near one-hot) -- gradients through softmax vanish when it's too peaked.
# Dividing by sqrt(head_size) keeps the variance of the scores roughly constant
# regardless of head_size, which keeps training stable.
wei = wei * (head_size ** -0.5)
print(f"wei shape: {wei.shape}")

### Mask the future, then softmax
Identical mechanism to Stage 3, Version 3 -- only the SOURCE of the scores has changed.

In [ ]:
# Same causal mask as every previous stage: block any position from seeing the future.
tril = torch.tril(torch.ones(T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))

# softmax turns the real, data-dependent scores into a genuine probability distribution
# per row -- but now, unlike Stage 3, the weights will NOT be uniform, because the
# underlying scores aren't uniform anymore.
wei = F.softmax(wei, dim=-1)

print("Attention weights for the FIRST token (can only see itself):")
print(wei[0, 0])   # expect [1, 0, 0, ..., 0]
print("\nAttention weights for the LAST token (sees all previous tokens, UNEQUALLY):")
print(wei[0, -1])  # data-dependent, NOT uniform -- compare to Stage 3's forced 1/T each

### Weighted aggregation of Values
The final step: use these learned, non-uniform weights to combine the VALUE vectors (not the raw
input `x`, and not the Query or Key vectors -- specifically the Value projection, which is "what gets
communicated" once a token has been attended to).

In [ ]:
# (B, T, T) @ (B, T, head_size) -> (B, T, head_size). Structurally identical to Stage
# 3's `wei @ x` -- the only difference is we're aggregating `v` (a learned projection)
# instead of the raw embeddings, and `wei` came from learned Q/K dot products instead
# of being hardcoded uniform.
out = wei @ v
print(f"Output shape: {out.shape}")

**Checkpoint -- what just happened, in one line each:**
1. **Query:** "what am I looking for?" (computed per token)
2. **Key:** "what do I contain?" (computed per token)
3. **Value:** "what do I actually communicate if you attend to me?" (computed per token)
4. **Attention:** Query . Key tells us how relevant each past token is to the current one
5. **Output:** a weighted sum of Values, weighted by that relevance

This single mechanism -- one "head" of self-attention -- is the fundamental building block of every
transformer, from this tiny model up through GPT-4 and beyond. Part 11 wraps this into a reusable
`Head` module, runs several of them in parallel (`MultiHeadAttention`), and combines it with a
feed-forward network into a full `Block` -- the repeating unit that gets stacked to build the actual
GPT model in Part 12.

**Stop here and run this section before continuing** -- confirm the first token's weights are exactly
`[1, 0, 0, ..., 0]` and the last token's weights are non-uniform, then we move to Part 11: building a
complete transformer block.

## Stage 5: Building a Complete Transformer Block (Part 11)

See `WALKTHROUGH.md` for the extended speed-dating analogy and real hand-verified numbers for each
piece. Four new pieces this stage: `Head` (wraps Stage 4's inline code into a reusable class),
`MultiHeadAttention` (several `Head`s in parallel), `FeedForward` (each token processes its own
gathered information), and `Block` (combines both with LayerNorm + residual connections -- the
repeating unit that gets stacked to build the actual GPT model next, in Part 12).

In [ ]:
# Hyperparameters used by every class below -- defined once here so Head/MultiHeadAttention
# don't need them passed in explicitly (matches the book's structure, where these are
# module-level globals set once before building the model).
n_embd = 32       # embedding width flowing through the whole model
n_head = 4         # how many attention heads run in parallel
block_size = 8      # context window (reused from Stage 1)
dropout = 0.0       # fraction of activations randomly zeroed during training (regularization);
                     # 0.0 here since we're not training yet -- real training uses ~0.1-0.2


class Head(nn.Module):
    """One head of self-attention -- IDENTICAL math to Stage 4, just wrapped as a reusable class
    so MultiHeadAttention below can create several of these at once."""
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        # register_buffer: stores the causal mask as part of the module's state (so it moves
        # to GPU with the rest of the model via .to(device)) WITHOUT treating it as a learnable
        # parameter -- it's a fixed constant, never updated by the optimizer.
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        v = self.value(x)
        # Same scaled dot-product attention as Stage 4, just written as C**-0.5 instead of
        # head_size**-0.5 -- equivalent here since C is the head's OWN internal dimension
        # at this point (the input width to this Head, not the outer model's n_embd).
        wei = q @ k.transpose(-2, -1) * (C ** -0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)   # randomly zero some attention weights during training --
                                    # prevents the model from over-relying on any single connection
        out = wei @ v
        return out


class MultiHeadAttention(nn.Module):
    """Several Heads running in PARALLEL (independent matchmakers, each specializing in a
    different kind of relevance), then combined."""
    def __init__(self, num_heads, head_size):
        super().__init__()
        # ModuleList (not a plain Python list!) -- required so PyTorch tracks each Head's
        # parameters properly (for .parameters(), optimizer updates, .to(device), etc.)
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        # After concatenating all heads back to n_embd width, one more linear layer lets
        # information MIX across heads -- without this, each head's output would stay in
        # its own separate "lane" forever, never combined into a unified representation.
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # Run every head on the SAME input x, independently, then concatenate their outputs
        # along the last dimension: num_heads outputs of width head_size -> one output of
        # width num_heads*head_size (== n_embd, by construction: head_size = n_embd // n_head).
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out


class FeedForward(nn.Module):
    """Each token processes its OWN gathered information, independently of every other
    token -- the 'going off alone to think about what you heard' step. No communication
    between positions happens here; that only happens in attention above."""
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),   # expand -- more "scratch space" to compute in;
                                                # the 4x ratio is a convention matching GPT-2
            nn.ReLU(),                         # nonlinearity -- without this, stacking two
                                                # Linear layers would collapse to one Linear layer
            nn.Linear(4 * n_embd, n_embd),   # contract back to the working width
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    """One full transformer block: attention (tokens communicate) + feed-forward (each
    token processes alone), each wrapped in LayerNorm + a residual connection. This is the
    unit that gets stacked N times to build the actual GPT model in Part 12."""
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head   # so num_heads * head_size concatenates back to n_embd
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        # PRE-norm structure (normalize BEFORE the sublayer, not after) -- more stable to
        # train than the original Transformer paper's post-norm design, and standard in
        # modern LLMs.
        #
        # x + self.sa(...): the residual connection. x is added back UNCHANGED alongside
        # whatever the attention sublayer computes -- so even if self.sa(...) is currently
        # weak or unhelpful (e.g. early in training), the original signal in x still gets
        # through. The sublayer only ever contributes an ADDITIVE refinement, never a
        # wholesale replacement.
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x


print("Head, MultiHeadAttention, FeedForward, and Block classes defined.")

### Verify shapes end to end

In [ ]:
torch.manual_seed(1337)
B, T = 4, 8
x = torch.randn(B, T, n_embd)

block = Block(n_embd, n_head)
out = block(x)
print(f"Input shape:  {x.shape}")
print(f"Output shape: {out.shape}")
print(f"Same shape in and out? {x.shape == out.shape}")
print("\nThis 'same shape in, same shape out' property is exactly what lets us STACK many")
print("Blocks in sequence (Part 12) -- each Block's output slots directly into the next")
print("Block's input, no reshaping needed anywhere.")

**Checkpoint:** a single `Block` takes `(B, T, n_embd)` in and returns `(B, T, n_embd)` out --
unchanged shape, but now every token's representation has communicated with earlier tokens
(multi-head attention) AND been individually refined (feed-forward), all while preserving the
original signal via residual connections. Stack several of these and you have the core of a real GPT
model.

**Stop here and run this section before continuing** -- confirm the input and output shapes match,
then we move to Part 12: assembling token + position embeddings and a stack of `Block`s into the
complete `GPTLanguageModel`.

## Stage 6: The Complete GPT Model (Part 12)

See `WALKTHROUGH.md` for the full breakdown, including a real crash demonstrating exactly why
`generate()` needs one new line this stage. Two new ideas: **position embeddings** (self-attention has
no built-in sense of order, so we add one) and **stacking Blocks** (works because every `Block`
preserves shape, verified back in Part 11).

In [ ]:
n_layer = 4   # NEW: how many Blocks get stacked -- this is the model's "depth"


class GPTLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        # Token embedding: same as every stage so far -- one row per vocabulary character.
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)

        # Position embedding: NEW. One row per POSITION (0, 1, 2, ... block_size-1), not per
        # character. WHY needed: self-attention is just weighted averaging over a set of
        # tokens -- nothing about it inherently knows "this token came first" vs "this token
        # came last." Without this, 'cat' at position 0 and 'cat' at position 5 would look
        # IDENTICAL to the attention mechanism. This gives every position a unique learned
        # vector to mix in, breaking that symmetry.
        self.position_embedding_table = nn.Embedding(block_size, n_embd)

        # Stack n_layer Blocks in sequence. This is valid ONLY because we verified in Part 11
        # that a Block's output shape exactly matches its input shape -- so each Block's
        # output slots directly into the next one's input, no adapter/reshape code needed.
        self.blocks = nn.Sequential(*[Block(n_embd, n_head) for _ in range(n_layer)])

        # One more LayerNorm after ALL the blocks -- stabilizes the final representation
        # before projecting it down to vocabulary-sized logits.
        self.ln_f = nn.LayerNorm(n_embd)

        # The final output layer: maps from the model's internal n_embd-wide representation
        # back OUT to vocab_size raw scores -- this is the piece that finally produces
        # "next character" predictions. Compare to the bigram model, where the embedding
        # table WAS directly the logits; here, many layers of processing happen first.
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok_emb = self.token_embedding_table(idx)                       # (B, T, C)
        pos_emb = self.position_embedding_table(torch.arange(T))        # (T, C) -- no batch dim!
        # Addition, not concatenation. pos_emb has no batch dimension, so PyTorch BROADCASTS
        # it across every sequence in the batch automatically -- the same T position vectors
        # get added onto every one of the B sequences, since "position 3" means the same
        # thing regardless of which sequence you're looking at.
        x = tok_emb + pos_emb                                             # (B, T, C)

        x = self.blocks(x)     # run through all n_layer Blocks in sequence
        x = self.ln_f(x)       # final stabilizing normalization
        logits = self.lm_head(x)   # (B, T, vocab_size)

        loss = None
        if targets is not None:
            # IDENTICAL flatten + cross_entropy pattern as the bigram model in Stage 2 --
            # the loss computation itself never changed, only what produces the logits did.
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            # NEW vs. the bigram model: crop to the last block_size tokens before each
            # forward pass. WHY this is now REQUIRED (it wasn't before): the bigram model
            # only ever used the single last character regardless of sequence length, so
            # feeding it a long sequence was harmless. This model's position_embedding_table
            # only has block_size rows -- asking it for a position beyond that crashes with
            # an IndexError. Cropping guarantees we never request a position it doesn't have.
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx


model = GPTLanguageModel()
n_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {n_params:,}")

### Verify shapes end to end, and see WHY the block_size crop is required

In [ ]:
xb, yb = get_batch('train')

# Inspect token vs position embedding shapes before they get added
tok_emb = model.token_embedding_table(xb)
pos_emb = model.position_embedding_table(torch.arange(xb.shape[1]))
print(f"tok_emb shape: {tok_emb.shape}  (B, T, C)")
print(f"pos_emb shape: {pos_emb.shape}  (T, C) -- no batch dimension, broadcasts across it")

logits, loss = model(xb, yb)
print(f"\nlogits shape: {logits.shape}")
print(f"loss: {loss.item():.4f}  (expected random baseline: {-torch.log(torch.tensor(1.0/vocab_size)):.4f})")

print("\nGeneration BEFORE any training (expect gibberish -- random weights):")
context = torch.zeros((1, 1), dtype=torch.long)
generated = model.generate(context, max_new_tokens=100)
print(decode(generated[0].tolist()))

In [ ]:
# Prove the crop is necessary: try running forward WITHOUT cropping on a
# deliberately-too-long sequence (longer than block_size).
try:
    long_idx = torch.zeros((1, block_size + 5), dtype=torch.long)
    model(long_idx)
    print("No error -- unexpected!")
except IndexError as e:
    print(f"Confirmed the crop is required -- without it: IndexError: {e}")
    print(f"position_embedding_table only has {block_size} rows (indices 0..{block_size-1}).")

**Checkpoint:** we now have a complete `GPTLanguageModel` -- token + position embeddings, a stack
of `n_layer` transformer `Block`s, a final LayerNorm, and an output head producing real vocabulary-sized
logits. Untrained, it's still gibberish (same as the untrained bigram model was) -- but the *architecture*
is now identical in structure to GPT-2, LLaMA, and every other decoder-only transformer; only the scale
differs from here on.

**Stop here and run this section before continuing** -- confirm the parameter count prints, shapes
match, and the IndexError reproduces as expected -- then we move to Part 13: actually training this
model on the full Shakespeare corpus, at real hyperparameter scale.